# Species Identification with Perch V2 

This notebook runs a complete species-identification pipeline using the
Perch V2 model downloaded from Kaggle.

## Workflow

1. **Download** Perch V2 from Kaggle with `kagglehub` (cached after the first run).
2. **Load** the TensorFlow SavedModel and its bundled list of species labels.
3. For each recording:
   - **load & resample** the audio to mono 32 kHz
   - **cut it into 5-second windows** (Perch's fixed input size)
   - **peak-normalize each window** — Perch's required preprocessing
   - **run the model** to get a score for every species
   - turn scores into **probabilities** (softmax) and keep the **top-k per window**.
4. **Filter** by a confidence threshold (one value, or a whole range) and **save**
   the detections as CSV.
5. Optionally **cut out audio clips** of the detections to listen to.

### Considerations
- Perch V2 is **not** amplitude-invariant. Before inference, each 5-second window must have its DC offset removed and its peak amplitude scaled to **0.25**, as Perch's official preprocessing. If you feed raw audio straight to the model you still get *an* answer, but the confidence scores come out systematically too low. The `peak_normalize` function below does exactly this step.

## 0 · Libraries

In [1]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # quiet TensorFlow's start-up logs

import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import tensorflow as tf

print("TensorFlow", tf.__version__, "— libraries ready.")

I0000 00:00:1784111195.040329    1918 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784111197.049968    1918 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow 2.21.0 — libraries ready.


## 1 · Download Perch V2 from Kaggle

`kagglehub` fetches the model the **first** time and caches it under
`~/.cache/kagglehub` for every run afterwards. The first download needs a Kaggle
API token in your environment (create one at Kaggle → *Settings* → *API*, then
`export KAGGLE_API_TOKEN=...` before launching Jupyter).

We use the **`perch_v2_cpu`** build, which runs on any machine. (The plain
`perch_v2` build is compiled for CUDA and will not run on a CPU.)


In [2]:
import kagglehub

MODEL_SLUG = "google/bird-vocalization-classifier/tensorFlow2/perch_v2_cpu"

model_path = Path(kagglehub.model_download(MODEL_SLUG))
print("Model files at:", model_path)

Model files at: /home/alex/.cache/kagglehub/models/google/bird-vocalization-classifier/tensorFlow2/perch_v2_cpu/1


## 2 · Load the model and the species labels

A Perch SavedModel is called through its `serving_default` signature: you hand it a
batch of 5-second windows shaped `[N, 160000]` (float32) and it returns, among
other things, a `label` array of shape `[N, 14795]` — one raw score (*logit*) per
species, per window.

The species names live in `assets/labels.csv` inside the downloaded model. Its
first line is a namespace tag, so we skip it; the remaining 14,795 names line up
one-to-one with the `label` scores.


In [3]:
model = tf.saved_model.load(str(model_path))
infer = model.signatures["serving_default"]   # inputs=[N, 160000] float32  ->  {"label", "embedding", ...}

SAMPLE_RATE    = 32_000                        # Perch's required input rate (Hz)
WINDOW_S       = 5.0                           # Perch's fixed window; do NOT change
WINDOW_SAMPLES = int(WINDOW_S * SAMPLE_RATE)   # 160000 samples per window

label_lines = (model_path / "assets" / "labels.csv").read_text().splitlines()
CLASS_NAMES = [ln.strip() for ln in label_lines[1:] if ln.strip()]   # skip the header line
print(f"Loaded {len(CLASS_NAMES)} species classes. First few: {CLASS_NAMES[:3]}")

Loaded 14795 species classes. First few: ['Abavorana luctuosa', 'Abeillia abeillei', 'Abroscopus albogularis']


## 3 · Preprocessing

Three small functions:

- **`load_audio`** — read any common audio file as mono and resample to 32 kHz.
- **`frame_windows`** — slice the recording into 5-second windows, stepping forward
  by `hop_s` seconds each time. Windows shorter than 5 s (a very short file) are
  padded; a leftover tail shorter than one window is dropped.
- **`peak_normalize`** — for each window, subtract the mean
  (remove DC offset) and scale so the loudest sample sits at 0.25.


In [4]:
def load_audio(path):
    """Load an audio file as mono float32 at Perch's 32 kHz rate."""
    audio, _ = librosa.load(str(path), sr=SAMPLE_RATE, mono=True)
    return audio.astype(np.float32)


def frame_windows(audio, hop_s):
    """Cut audio into Perch's 5 s windows, stepping by hop_s seconds.

    Returns an array of shape [n_windows, 160000].
    """
    hop_samples = int(hop_s * SAMPLE_RATE)
    if len(audio) < WINDOW_SAMPLES:
        audio = librosa.util.pad_center(audio, size=WINDOW_SAMPLES, axis=-1)
    frames = librosa.util.frame(
        audio, frame_length=WINDOW_SAMPLES, hop_length=hop_samples, axis=-1
    ).swapaxes(-1, -2)
    return frames


def peak_normalize(frames, target_peak=0.25):
    """Perch's required per-window preprocessing: remove DC, scale peak to 0.25.

    Perch V2 is NOT amplitude-invariant, so skipping this yields wrong (too-low)
    confidence scores. Silent windows (peak 0) are left at zero.
    """
    frames = frames.astype(np.float32).copy()
    frames -= frames.mean(axis=-1, keepdims=True)
    peak = np.max(np.abs(frames), axis=-1, keepdims=True)
    frames = np.divide(frames, peak, out=np.zeros_like(frames), where=peak > 0.0)
    return frames * target_peak

## 4 · Inference 

`window_probabilities` runs the model on a batch of windows and converts the raw
scores into **probabilities** with softmax.

`identify_file` ties it together for one recording and returns a table with the
top-k species for every window. 

In [5]:
def window_probabilities(frames):
    """Run Perch on a batch of windows -> softmax probabilities [n_windows, n_classes]."""
    logits = infer(inputs=tf.constant(frames, dtype=tf.float32))["label"].numpy()
    return tf.nn.softmax(logits, axis=-1).numpy()


def identify_file(path, hop_s, top_k):
    """Full per-file pipeline -> DataFrame of the top-k predictions per window."""
    audio = load_audio(path)
    frames = frame_windows(audio, hop_s)
    frames = peak_normalize(frames)             # <-- required step, BEFORE inference
    probs = window_probabilities(frames)        # [n_windows, n_classes]

    rows = []
    for w, prob in enumerate(probs):
        start_s = w * hop_s
        top_idx = np.argsort(prob)[::-1][:top_k]
        for rank, idx in enumerate(top_idx, start=1):
            rows.append({
                "file": str(path),
                "start_s": round(start_s, 3),
                "end_s": round(start_s + WINDOW_S, 3),
                "window_s": WINDOW_S,
                "hop_s": hop_s,
                "rank": rank,
                "label": CLASS_NAMES[idx],
                "confidence": float(prob[idx]),
            })
    return pd.DataFrame(rows)

## 5 · Choose your input & output folders

- **`INPUT_DIR`** — a folder of recordings (searched recursively; `.wav .flac .ogg
  .mp3 .m4a .aiff .aif` are accepted).
- **`OUTPUT_DIR`** — where results are written (created automatically).


In [6]:
INPUT_DIR  = Path("/home/alex/projects/PerchLab/data/picampall/Dades")            # <-- change me: your recordings folder
OUTPUT_DIR = Path("/home/alex/projects/PerchLab/data/outputs")       # <-- change me (optional): where results go

AUDIO_EXTS = {".wav", ".flac", ".ogg", ".mp3", ".m4a", ".aiff", ".aif"}

def discover_audio(root):
    """Return sorted audio files anywhere under `root`."""
    return sorted(p for p in Path(root).rglob("*") if p.suffix.lower() in AUDIO_EXTS)

Confirm the files are found before running the model.


In [7]:
if not INPUT_DIR.exists():
    print(f"⚠ Input folder not found: {INPUT_DIR.resolve()}")
    print("  Point INPUT_DIR at a real folder of recordings and re-run this cell.")
else:
    found = discover_audio(INPUT_DIR)
    print(f"Found {len(found)} audio file(s) under {INPUT_DIR.resolve()}")
    for f in found[:10]:
        print("  -", f.name)

Found 30 audio file(s) under /home/alex/projects/PerchLab/data/picampall/Dades
  - PIC02_20250615_045602.wav
  - PIC02_20250615_055602.wav
  - PIC02_20250616_045602.wav
  - PIC02_20250616_055602.wav
  - PIC02_20250617_045602.wav
  - PIC02_20250617_055602.wav
  - PIC02_20250618_045602.wav
  - PIC02_20250618_055602.wav
  - PIC02_20250619_045602.wav
  - PIC02_20250619_055602.wav


## 6 · Hop size and Top-K species

- **`HOP_S`** — how far the 5-second window slides each step. `5.0` gives
  back-to-back windows, while a smaller value overlaps them.
- **`TOP_K`** — how many of the highest-scoring species to keep per window.

In [8]:
HOP_S = 5.0   # step between 5 s windows (s). < 5.0 overlaps them
TOP_K = 3     # keep this many highest-scoring species per window

## 7 · Confidence threshold(s)

A prediction becomes a *detection* when its confidence is **≥** the threshold.

A **sweep** can be triggered True/False in order to test multiple thresholds, getting one output folder for each one.

In [9]:
RUN_SWEEP = False    # False = one threshold; True = a whole range

THRESHOLD = 0.1                              # used when RUN_SWEEP is False
SWEEP_START, SWEEP_END, SWEEP_STEP = 0.1, 1.0, 0.1   # used when RUN_SWEEP is True

def threshold_values():
    """The list of thresholds to export at."""
    if not RUN_SWEEP:
        return [THRESHOLD]
    values, v = [], SWEEP_START
    while v <= SWEEP_END + 1e-9:
        values.append(round(v, 6))
        v += SWEEP_STEP
    return values

## 8 · Extract audio segments (optional)

To verify detections, we can cut a short clip around each recording. Rather than
export *every* detection, we sample a balanced spread across confidence: the 0–1
confidence range is split into equal-width bands named **bins** (`BIN_WIDTH`), and at most
`MAX_PER_BIN` clips are kept **per species and per band**. Clips are saved into
one subfolder per species.

- **`CLIP_DURATION_S`** — length of the central clip.
- **`CONTEXT_S`** — extra seconds added on **each** side. a 5 s clip
  with `CONTEXT_S = 1.0` becomes a 7 s clip (1 s before + 5 s + 1 s after).
- **`BIN_WIDTH`** — width of each confidence band (0.1 → bands 0.0–0.1, 0.1–0.2, …).
- **`MAX_PER_BIN`** — max clips kept per species, per band.
- **`SEGMENT_SEED`** — makes the random sampling reproducible.


In [10]:
EXTRACT_SEGMENTS = False   # set True to also export audio clips of detections

CLIP_DURATION_S = 5.0   # length of the central clip (s)
CONTEXT_S       = 0.0   # extra seconds on EACH side ("wings")
BIN_WIDTH       = 0.1   # width of each confidence band (0.1 -> 0.0-0.1, 0.1-0.2, ...)
MAX_PER_BIN     = 20    # max clips kept per species, per confidence band
SEGMENT_SEED    = 0     # random seed for the sampling


def extract_segments(detections, out_dir):
    """Sample detections evenly across confidence, separately for each species.

    Each detection is placed in a confidence band (its score // BIN_WIDTH). We then
    group by (species, band) and keep at most MAX_PER_BIN clips from each group.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    rng = random.Random(SEGMENT_SEED)
    total_s = CLIP_DURATION_S + 2.0 * max(0.0, CONTEXT_S)
    target = int(round(total_s * SAMPLE_RATE))

    dets = detections.copy()
    dets["_bin"] = (dets["confidence"] / BIN_WIDTH).astype(int)   # confidence band index
    chosen = []
    for _, grp in dets.groupby(["label", "_bin"]):                # per species, per band
        idx = list(grp.index)
        chosen += rng.sample(idx, MAX_PER_BIN) if len(idx) > MAX_PER_BIN else idx

    written = 0
    for i in chosen:
        row = dets.loc[i]
        center = 0.5 * (row["start_s"] + row["end_s"])
        start = max(0.0, center - total_s / 2.0)
        clip, _ = librosa.load(
            str(row["file"]), sr=SAMPLE_RATE, mono=True, offset=start, duration=total_s
        )
        clip = clip.astype(np.float32)
        if len(clip) < target:
            clip = np.pad(clip, (0, target - len(clip)))
        clip = clip[:target]

        species = re.sub(r"[^\w.-]+", "_", str(row["label"])).strip("_") or "unknown"
        species_dir = out_dir / species
        species_dir.mkdir(parents=True, exist_ok=True)
        name = f'{Path(row["file"]).stem}_{species}_{row["confidence"]:.3f}_{int(row["start_s"])}s.wav'
        sf.write(species_dir / name, clip, SAMPLE_RATE)
        written += 1
    return written

## 9 · Run the model over every recording

Loads and scores each file once. Progress prints per file. The result, `predictions`, holds the top-k species for every window of every recording, before any threshold is applied.


In [11]:
files = discover_audio(INPUT_DIR)
if not files:
    raise SystemExit(f"No audio files found under {INPUT_DIR}")

per_file = []
for i, path in enumerate(files, start=1):
    print(f"[{i}/{len(files)}] {path.name}")
    per_file.append(identify_file(path, HOP_S, TOP_K))

predictions = pd.concat(per_file, ignore_index=True)
print(f"Done: {len(predictions)} window-predictions from {len(files)} file(s).")

[1/30] PIC02_20250615_045602.wav
[2/30] PIC02_20250615_055602.wav
[3/30] PIC02_20250616_045602.wav
[4/30] PIC02_20250616_055602.wav
[5/30] PIC02_20250617_045602.wav
[6/30] PIC02_20250617_055602.wav
[7/30] PIC02_20250618_045602.wav
[8/30] PIC02_20250618_055602.wav
[9/30] PIC02_20250619_045602.wav
[10/30] PIC02_20250619_055602.wav
[11/30] PIC02_20250620_045702.wav
[12/30] PIC02_20250620_055702.wav
[13/30] PIC02_20250621_045702.wav
[14/30] PIC02_20250621_055702.wav
[15/30] PIC02_20250622_045702.wav
[16/30] PIC02_20250622_055702.wav
[17/30] PIC03_20250615_045602.wav
[18/30] PIC03_20250615_055602.wav
[19/30] PIC03_20250616_045602.wav
[20/30] PIC03_20250616_055602.wav
[21/30] PIC03_20250617_045602.wav
[22/30] PIC03_20250617_055602.wav
[23/30] PIC03_20250618_045602.wav
[24/30] PIC03_20250618_055602.wav
[25/30] PIC03_20250619_045602.wav
[26/30] PIC03_20250619_055602.wav
[27/30] PIC03_20250620_045702.wav
[28/30] PIC03_20250620_055702.wav
[29/30] PIC03_20250621_045702.wav
[30/30] PIC03_20250621_

## 10 · Apply the threshold(s) and save the output

For each threshold, we keep the predictions at or above it. Also creates a
`threshold_<value>/` folder: one CSV per recording, a combined `all_detections.csv`,
and a `summary.txt` of per-species counts. Also a `manifest.json` records the settings.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
counts_by_threshold = {}

for t in threshold_values():
    kept = predictions[predictions["confidence"] >= t].copy()
    kept["threshold"] = t
    kept = kept[[c for c in kept.columns if c != "file"] + ["file"]]   # file path as the last column

    thr_dir = OUTPUT_DIR / f"threshold_{t:.2f}"
    thr_dir.mkdir(parents=True, exist_ok=True)
    for fname, grp in kept.groupby("file"):
        grp.to_csv(thr_dir / f"{Path(fname).stem}.csv", index=False)
    kept.to_csv(thr_dir / "all_detections.csv", index=False)
    (thr_dir / "summary.txt").write_text(
        f"Threshold {t:.2f}\nDetections: {len(kept)}\n\n"
        + (kept["label"].value_counts().to_string() if len(kept) else "(none)"),
        encoding="utf-8",
    )
    counts_by_threshold[f"{t:.2f}"] = len(kept)

manifest = {
    "model_slug": MODEL_SLUG,
    "sample_rate": SAMPLE_RATE,
    "window_s": WINDOW_S,
    "hop_s": HOP_S,
    "top_k": TOP_K,
    "thresholds": threshold_values(),
    "n_classes": len(CLASS_NAMES),
    "n_files": len(files),
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Detections per threshold:", counts_by_threshold)

### Extract the clips (if enabled)


In [13]:
if EXTRACT_SEGMENTS:
    primary = threshold_values()[0]
    dets = pd.read_csv(OUTPUT_DIR / f"threshold_{primary:.2f}" / "all_detections.csv")
    n = extract_segments(dets, OUTPUT_DIR / f"threshold_{primary:.2f}" / "segments")
    print(f"Wrote {n} clip(s) under threshold_{primary:.2f}/segments/")
else:
    print("Segment extraction is off (EXTRACT_SEGMENTS = False).")

Segment extraction is off (EXTRACT_SEGMENTS = False).


## 11 · Explore the results

Load the combined table for the first threshold, preview it, and count species.


In [14]:
primary = threshold_values()[0]
agg_csv = OUTPUT_DIR / f"threshold_{primary:.2f}" / "all_detections.csv"
detections_df = pd.read_csv(agg_csv)
print(f"{len(detections_df)} detections at threshold {primary:.2f}  ->  {agg_csv}")
detections_df.head(15)

1914 detections at threshold 0.10  ->  /home/alex/projects/PerchLab/data/outputs/threshold_0.10/all_detections.csv


,file,start_s,end_s,window_s,hop_s,rank,label,confidence,threshold
0,/home/alex/projects/PerchLab/data/picampall/Da...,5.0,10.0,5.0,5.0,1,Acrocephalus scirpaceus,0.522375,0.1
1,/home/alex/projects/PerchLab/data/picampall/Da...,10.0,15.0,5.0,5.0,1,Acrocephalus scirpaceus,0.466570,0.1
2,/home/alex/projects/PerchLab/data/picampall/Da...,15.0,20.0,5.0,5.0,1,Acrocephalus scirpaceus,0.110577,0.1
3,/home/alex/projects/PerchLab/data/picampall/Da...,20.0,25.0,5.0,5.0,1,Acrocephalus scirpaceus,0.126882,0.1
4,/home/alex/projects/PerchLab/data/picampall/Da...,30.0,35.0,5.0,5.0,1,Acrocephalus scirpaceus,0.151625,0.1
5,/home/alex/projects/PerchLab/data/picampall/Da...,35.0,40.0,5.0,5.0,1,Acrocephalus scirpaceus,0.605051,0.1
6,/home/alex/projects/PerchLab/data/picampall/Da...,40.0,45.0,5.0,5.0,1,Acrocephalus scirpaceus,0.591398,0.1
7,/home/alex/projects/PerchLab/data/picampall/Da...,45.0,50.0,5.0,5.0,1,Acrocephalus scirpaceus,0.167383,0.1
8,/home/alex/projects/PerchLab/data/picampall/Da...,45.0,50.0,5.0,5.0,2,Acrocephalus arundinaceus,0.115715,0.1
9,/home/alex/projects/PerchLab/data/picampall/Da...,55.0,60.0,5.0,5.0,1,Acrocephalus scirpaceus,0.472627,0.1


In [15]:
if len(detections_df):
    print(detections_df["label"].value_counts().head(20))
else:
    print("No detections at this threshold — try a lower one.")

label
Acrocephalus scirpaceus       594
Acrocephalus arundinaceus     481
Cisticola juncidis            359
Locustella luscinioides        85
Nycticorax nycticorax          60
Tachybaptus ruficollis         36
Troglodytes troglodytes        33
Tringa ochropus                17
Fulica atra                    16
Anas platyrhynchos             16
Himantopus himantopus          14
Acrocephalus palustris         14
Gallinula chloropus            13
Tringa nebularia               10
Chroicocephalus ridibundus      8
Phylloscopus trochilus          8
Hirundo rustica                 7
Anthus hodgsoni                 7
Acrocephalus schoenobaenus      7
Actitis hypoleucos              6
Name: count, dtype: int64


## 12 · Optimal Confidence Threshold Detection

Sections 5–11 gave every window a confidence score — but *how high* does a score
have to be before you trust it? That cutoff is species-specific: a 0.3 for one
bird can be as reliable as a 0.7 for another. This section estimates, **per
species**, the confidence threshold at which a detection has a chosen probability
(default **95 %**) of being correct. It reproduces `perchlab`'s 4th menu option
("Optimal Confidence Threshold Detection") with no project code.

**Method** (Wood et al. 2023; validated against Bota et al. 2023):

1. Start from **human-validated** detections — clips a person sorted into
   `correct/` and `incorrect/` folders.
2. Run Perch on each clip and recover the confidence it gives the **target
   species** (the strongest window).
3. Back-transform confidence to the logit scale `L = ln(c / (1 − c))` and fit a
   logistic regression `P(correct) = sigmoid(b0 + b1·L)`.
4. Invert it for the confidence `c*` where `P(correct) = target`:
   `L* = (logit(target) − b0) / b1`, then `c* = sigmoid(L*)`.

Outputs: a **precision table** and a **probability-of-correct plot** per species.

### Your validated dataset & options

Point `VALID_DIR` at a folder of sorted clips. Two layouts are accepted:

**One species** — `correct/` and `incorrect/` directly inside `VALID_DIR` (then
set `TARGET_SPECIES` to that bird's scientific name):

```
VALID_DIR/
  correct/     PIC02_..._Acrocephalus scirpaceus_0.62_15s.wav
  incorrect/   ...
```

**Several species** — one folder per scientific name, each with its own
`correct/` + `incorrect/` (leave `TARGET_SPECIES = None`):

```
VALID_DIR/
  Acrocephalus scirpaceus/{correct,incorrect}/...
  Cettia cetti/{correct,incorrect}/...
```

These are typically the clips exported by §8 (`EXTRACT_SEGMENTS`), listened to,
and dragged into the right folder. Folder names are case-insensitive
(`correct/present/true/tp/yes` = correct; `incorrect/absent/false/fp/no` =
incorrect). The options below mirror the CLI's `threshold` knobs.

In [ ]:
# --- where the validated clips live -----------------------------------------
VALID_DIR = Path("/home/alex/projects/PerchLab/data/validated")   # <-- change me

# --- detection options (mirror perchlab's 4th menu option) ------------------
TARGET_SPECIES     = None       # scientific name for the one-species layout; None = one per subfolder
TARGET_PROBABILITY = 0.95       # probability of "correct" the threshold is solved for
BIN_EDGES          = [0.1, 0.3, 0.5]   # lower edges of the precision-table confidence bands
ACTIVATION         = "softmax"  # "softmax" (default, matches §4) or "sigmoid" (per-class)
MIN_SAMPLES        = 8          # fewest validated clips needed before a species is fitted

THRESHOLD_OUTPUT_DIR = OUTPUT_DIR / "optimal_threshold"   # where the table/plot/report go

### Load the validated clips

Walk `VALID_DIR`, read each clip's verdict from the folder it sits in, and preview
how many correct/incorrect clips were found per species.

In [ ]:
CORRECT_NAMES   = {"correct", "present", "true", "tp", "1", "yes", "positive"}
INCORRECT_NAMES = {"incorrect", "absent", "false", "fp", "0", "no", "negative"}


def _verdict(name):
    """Map a folder name to True (correct) / False (incorrect) / None (neither)."""
    key = name.strip().lower()
    if key in CORRECT_NAMES:
        return True
    if key in INCORRECT_NAMES:
        return False
    return None


def _has_verdict_subdirs(directory):
    """True if `directory` has at least one correct/ or incorrect/ subfolder."""
    return any(_verdict(d.name) is not None for d in Path(directory).iterdir() if d.is_dir())


def _collect_species(species_dir, species):
    """Every validated clip under one species folder, tagged correct/incorrect."""
    found = []
    for sub in sorted(Path(species_dir).iterdir()):
        verdict = _verdict(sub.name) if sub.is_dir() else None
        if verdict is None:
            continue
        for path in discover_audio(sub):          # discover_audio defined in §5
            found.append({"path": path, "species": species, "correct": verdict})
    return found


def load_validated_dataset(root, species=None):
    """Discover validated clips under `root` (single- or multi-species layout)."""
    root = Path(root)
    if not root.is_dir():
        raise SystemExit(f"Validated dataset folder not found: {root}")
    if _has_verdict_subdirs(root):                # one-species layout
        if not species:
            raise SystemExit("Found correct/incorrect at the top level; set TARGET_SPECIES.")
        files = _collect_species(root, species)
    else:                                         # multi-species layout
        files = []
        for species_dir in sorted(p for p in root.iterdir() if p.is_dir()):
            if not _has_verdict_subdirs(species_dir):
                print(f"  skipping '{species_dir.name}': no correct/incorrect subfolders")
                continue
            files += _collect_species(species_dir, species or species_dir.name)
    if not files:
        raise SystemExit(f"No validated clips found under {root} (need correct/ and incorrect/).")
    return files


validated = load_validated_dataset(VALID_DIR, TARGET_SPECIES)
val_df = pd.DataFrame([{"species": f["species"], "correct": f["correct"]} for f in validated])
print(f"Loaded {len(validated)} validated clips across {val_df['species'].nunique()} species:")
print(val_df.groupby("species")["correct"].agg(n="size", correct="sum"))

### Score each clip with Perch

For each clip, run Perch (reusing §3–§4) and keep the confidence it gives the
**target species** — taking the strongest window, the one that drove the
detection. `ACTIVATION` picks softmax (default, as in §4) or per-class sigmoid.

In [ ]:
CLASS_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}


def window_logits(frames):
    """Raw Perch logits for the windows, scored in batches (like §4)."""
    out = []
    for start in range(0, len(frames), BATCH_WINDOWS):
        batch = frames[start:start + BATCH_WINDOWS]
        out.append(infer(inputs=tf.constant(batch, dtype=tf.float32))["label"].numpy())
    return np.concatenate(out, axis=0)


def clip_target_confidence(path, species_idx, activation):
    """Perch's confidence for one species on a clip = its strongest window's score."""
    frames = peak_normalize(frame_windows(load_audio(path), HOP_S))
    logits = window_logits(frames)
    if activation == "softmax":
        conf = tf.nn.softmax(logits, axis=-1).numpy()[:, species_idx]
    else:  # per-class sigmoid
        conf = 1.0 / (1.0 + np.exp(-np.clip(logits[:, species_idx], -700.0, 700.0)))
    return float(np.max(conf))


scores_by_species = {}          # species -> {"confidence": [...], "correct": [...]}
for j, f in enumerate(validated, start=1):
    idx = CLASS_INDEX.get(f["species"])
    if idx is None:
        raise SystemExit(f"'{f['species']}' is not a Perch class; use the scientific name.")
    print(f"[{j}/{len(validated)}] {f['species']} · {Path(f['path']).name}")
    bucket = scores_by_species.setdefault(f["species"], {"confidence": [], "correct": []})
    bucket["confidence"].append(clip_target_confidence(f["path"], idx, ACTIVATION))
    bucket["correct"].append(bool(f["correct"]))

print("Scored:", {s: len(v["correct"]) for s, v in scores_by_species.items()})

### Fit the logistic model per species

The statistics below take arrays (confidence + correctness), so they run without
the model. `fit_species_threshold` fits `P(correct) = sigmoid(b0 + b1·L)`, solves
for the target-probability threshold, builds the precision bins, and computes the
fitted curve + 95 % band for the plot. Species with too few clips, all-correct /
all-incorrect data, or a non-positive slope are reported but not fitted.

In [ ]:
from sklearn.linear_model import LogisticRegression

EPS = 1e-6


def logit_score(confidence):
    """Standard logit L = ln(c / (1 - c)) (inverse sigmoid), clipped to stay finite."""
    c = np.clip(np.asarray(confidence, dtype=np.float64), EPS, 1.0 - EPS)
    return np.log(c / (1.0 - c))


def sigmoid(eta):
    """Logistic sigmoid, argument clipped to avoid exp overflow."""
    return 1.0 / (1.0 + np.exp(-np.clip(eta, -700.0, 700.0)))


def precision_bins(confidence, correct, bin_edges):
    """Bin detections by confidence; count verified (correct) per band."""
    conf = np.asarray(confidence, dtype=np.float64)
    ok = np.asarray(correct).astype(bool)
    edges = list(bin_edges) + [1.0 + EPS]
    rows = []
    for i in range(len(edges) - 1):
        low, high = edges[i], edges[i + 1]
        in_bin = (conf >= low) & (conf < high)
        is_last = i == len(edges) - 2
        disp_high = min(high, 1.0)
        det = int(in_bin.sum())
        ver = int((in_bin & ok).sum())
        rows.append({
            "category": f"[{low:.2f}, {disp_high:.2f}{']' if is_last else ')'}",
            "detections": det, "verified": ver,
            "precision": ver / det if det else float("nan"),
        })
    return rows


def _fit_logistic(logit, correct):
    """Unregularised MLE fit; returns (intercept, slope, covariance)."""
    model = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=1000)
    model.fit(logit.reshape(-1, 1), correct.astype(int))
    b0, b1 = float(model.intercept_[0]), float(model.coef_[0][0])
    design = np.column_stack([np.ones_like(logit), logit])
    p = sigmoid(b0 + b1 * logit)
    w = np.clip(p * (1.0 - p), 1e-12, None)
    fisher = design.T @ (design * w[:, None])
    try:
        cov = np.linalg.inv(fisher)
    except np.linalg.LinAlgError:
        cov = np.linalg.pinv(fisher)
    return b0, b1, cov


def fit_species_threshold(species, confidence, correct, target_probability=0.95,
                          bin_edges=(0.1, 0.3, 0.5), min_samples=8):
    """Fit the logistic model and solve for the target-probability threshold."""
    conf = np.clip(np.asarray(confidence, dtype=np.float64), 0.0, 1.0)
    ok = np.asarray(correct).astype(bool)
    n, n_correct = int(conf.size), int(ok.sum())
    n_incorrect = n - n_correct
    result = {
        "species": species, "n": n, "n_correct": n_correct, "n_incorrect": n_incorrect,
        "target_probability": target_probability, "threshold": float("nan"),
        "intercept": float("nan"), "slope": float("nan"), "fitted": False, "note": "",
        "bins": precision_bins(conf, ok, bin_edges),
        "points_conf": conf, "points_correct": ok.astype(float),
        "curve_conf": np.empty(0), "curve_prob": np.empty(0),
        "curve_lo": np.empty(0), "curve_hi": np.empty(0),
    }
    if n < min_samples:
        result["note"] = f"only {n} validated detections (need >= {min_samples})"
        return result
    if n_correct == 0 or n_incorrect == 0:
        only = "correct" if n_incorrect == 0 else "incorrect"
        result["note"] = f"all detections are {only}; cannot fit a regression"
        return result

    logit = logit_score(conf)
    b0, b1, cov = _fit_logistic(logit, ok)
    result["intercept"], result["slope"] = b0, b1
    if b1 <= 0:
        result["note"] = "confidence not positively associated with correctness (slope <= 0)"
        return result

    target_logit = float(np.log(target_probability / (1.0 - target_probability)))
    l_star = (target_logit - b0) / b1
    threshold = float(sigmoid(l_star))              # sigmoid => always in (0, 1)
    result["threshold"], result["fitted"] = threshold, True
    if threshold >= conf.max():
        result["note"] = "threshold exceeds the largest validated confidence (extrapolated)"
    elif threshold <= conf.min():
        result["note"] = "target met below the smallest validated confidence (extrapolated)"

    # Fitted probability curve + 95% band (delta method) for the plot.
    grid = np.linspace(EPS, 1.0 - EPS, 200)
    glogit = logit_score(grid)
    design = np.column_stack([np.ones_like(glogit), glogit])
    eta = b0 + b1 * glogit
    se = np.sqrt(np.clip(np.einsum("ij,jk,ik->i", design, cov, design), 0.0, None))
    result["curve_conf"], result["curve_prob"] = grid, sigmoid(eta)
    result["curve_lo"], result["curve_hi"] = sigmoid(eta - 1.96 * se), sigmoid(eta + 1.96 * se)
    return result


threshold_results = []
for species in sorted(scores_by_species):
    conf = np.asarray(scores_by_species[species]["confidence"], dtype=np.float64)
    correct = np.asarray(scores_by_species[species]["correct"], dtype=bool)
    r = fit_species_threshold(species, conf, correct, target_probability=TARGET_PROBABILITY,
                              bin_edges=BIN_EDGES, min_samples=MIN_SAMPLES)
    threshold_results.append(r)
    if r["fitted"]:
        print(f"{species}: threshold = {r['threshold']:.3f}  (n={r['n']}, {r['n_correct']} correct)"
              + (f"  — {r['note']}" if r["note"] else ""))
    else:
        print(f"{species}: no threshold — {r['note']}")

### Tables: estimated thresholds & precision by confidence band

Two tables, saved to `THRESHOLD_OUTPUT_DIR` and shown inline: the per-species
thresholds, and the Table-1-style precision breakdown per confidence band.

In [ ]:
THRESHOLD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- thresholds table (one row per species) ---------------------------------
thr_rows = [{
    "species": r["species"],
    "threshold": round(r["threshold"], 4) if r["fitted"] else None,
    "target_probability": r["target_probability"],
    "n": r["n"], "n_correct": r["n_correct"], "n_incorrect": r["n_incorrect"],
    "total_precision_pct": round(100 * r["n_correct"] / r["n"], 1) if r["n"] else None,
    "intercept": round(r["intercept"], 4) if r["fitted"] else None,
    "slope": round(r["slope"], 4) if r["fitted"] else None,
    "fitted": r["fitted"], "note": r["note"],
} for r in threshold_results]
thresholds_df = pd.DataFrame(thr_rows)
thresholds_df.to_csv(THRESHOLD_OUTPUT_DIR / "thresholds.csv", index=False)
(THRESHOLD_OUTPUT_DIR / "thresholds.json").write_text(json.dumps(thr_rows, indent=2), encoding="utf-8")

# --- precision-by-confidence-band table (Table 1 style) ---------------------
prec_rows = []
for r in threshold_results:
    for b in r["bins"]:
        prec_rows.append({
            "species": r["species"], "confidence_category": b["category"],
            "detections": b["detections"], "verified": b["verified"],
            "precision_pct": round(100 * b["precision"], 1) if b["detections"] else None,
        })
    tot_det = sum(b["detections"] for b in r["bins"])
    tot_ver = sum(b["verified"] for b in r["bins"])
    prec_rows.append({
        "species": r["species"], "confidence_category": "TOTAL",
        "detections": tot_det, "verified": tot_ver,
        "precision_pct": round(100 * tot_ver / tot_det, 1) if tot_det else None,
    })
precision_df = pd.DataFrame(prec_rows)
precision_df.to_csv(THRESHOLD_OUTPUT_DIR / "precision_table.csv", index=False)

print(f"Wrote thresholds.csv, thresholds.json, precision_table.csv -> {THRESHOLD_OUTPUT_DIR}")
display(thresholds_df)
display(precision_df)

### Plot: probability of a correct detection

One panel per species: the validated clips as a 0/1 scatter, the fitted logistic
curve with its 95 % band, the target-probability line, and the estimated
threshold where the curve crosses it.

In [ ]:
import matplotlib.pyplot as plt


def plot_probability_curves(results, path):
    """Per-species panel: 0/1 scatter, fitted curve + 95% band, target & threshold lines."""
    n = max(1, len(results))
    ncols = min(2, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.2 * nrows), squeeze=False)
    rng = np.random.default_rng(0)
    flat = axes.ravel()
    for ax, r in zip(flat, results, strict=False):
        jitter = rng.uniform(-0.03, 0.03, size=r["points_correct"].shape)
        ax.scatter(r["points_conf"], r["points_correct"] + jitter, s=10, color="black",
                   alpha=0.5, zorder=3)
        if r["fitted"] and r["curve_conf"].size:
            ax.plot(r["curve_conf"], r["curve_prob"], color="tab:blue", lw=2, zorder=2)
            ax.fill_between(r["curve_conf"], r["curve_lo"], r["curve_hi"],
                            color="tab:blue", alpha=0.15, zorder=1)
            ax.axhline(r["target_probability"], color="gray", ls=":", lw=1)
            if np.isfinite(r["threshold"]):
                ax.axvline(r["threshold"], color="tab:red", ls="--", lw=1.5, zorder=4)
                ax.annotate(f"threshold = {r['threshold']:.2f}", xy=(r["threshold"], 0.5),
                            xytext=(6, 0), textcoords="offset points", color="tab:red",
                            fontsize=8, rotation=90, va="center")
        ax.set_xlim(0, 1)
        ax.set_ylim(-0.08, 1.08)
        ax.set_xlabel("Confidence score")
        ax.set_ylabel("Prob. correct detection")
        ax.set_title(r["species"] if r["fitted"] else f"{r['species']} (no fit)", fontsize=10)
        ax.grid(True, alpha=0.25)
    for ax in flat[len(results):]:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    return fig


fig = plot_probability_curves(threshold_results, THRESHOLD_OUTPUT_DIR / "probability_curves.png")
plt.show()
print(f"Saved plot -> {THRESHOLD_OUTPUT_DIR / 'probability_curves.png'}")

### Optional: a `report.md`

Writes the same human-readable report the CLI produces, bundling both tables and
the figure into one Markdown file.

In [ ]:
target = threshold_results[0]["target_probability"] if threshold_results else TARGET_PROBABILITY
lines = [
    "# PerchLab — Optimal Confidence Threshold", "",
    f"Confidence threshold for a **{target:.0%} probability of correct identification**, "
    "estimated per species by logistic regression of human-validated detections on the "
    "logit-transformed confidence score.", "",
    "## Estimated thresholds", "",
    "| Species | Threshold | n | Correct | Incorrect | Overall precision |",
    "| --- | --- | --- | --- | --- | --- |",
]
for r in threshold_results:
    thr = f"**{r['threshold']:.2f}**" if r["fitted"] else "—"
    prec = f"{100 * r['n_correct'] / r['n']:.1f}%" if r["n"] else "—"
    lines.append(f"| {r['species']} | {thr} | {r['n']} | {r['n_correct']} | {r['n_incorrect']} | {prec} |")
    if r["note"]:
        lines.append(f"|   ↳ *{r['note']}* | | | | | |")
lines += ["", "## Precision by confidence category", ""]
for r in threshold_results:
    lines += [f"### {r['species']}", "",
              "| Confidence category | Detections | Verified | Precision |",
              "| --- | --- | --- | --- |"]
    for b in r["bins"]:
        prec = f"{100 * b['precision']:.1f}%" if b["detections"] else "—"
        lines.append(f"| {b['category']} | {b['detections']} | {b['verified']} | {prec} |")
    tot_det = sum(b["detections"] for b in r["bins"])
    tot_ver = sum(b["verified"] for b in r["bins"])
    tot_prec = f"{100 * tot_ver / tot_det:.1f}%" if tot_det else "—"
    lines += [f"| **TOTAL** | {tot_det} | {tot_ver} | {tot_prec} |", ""]
lines += ["## Figure", "", "![Probability of correct detection](probability_curves.png)", ""]
(THRESHOLD_OUTPUT_DIR / "report.md").write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote report.md -> {THRESHOLD_OUTPUT_DIR / 'report.md'}")